In [22]:
import pandas as pd 
import gzip
import jsonlines
import urllib
from unidecode import unidecode
from tqdm.auto import tqdm

def parse_path(url):
    path = url.split(')')[-1]
    path = path.split('sfchronicle.com')[-1]
    path = urllib.parse.unquote(path, encoding='utf-8', errors='replace')
    path = unidecode(path).replace(' ', '')
    return path.lower()

In [31]:
fetched_article_cache_path = 'cache/full-article-df-for-matching.csv.gz'
if not os.path.exists(fetched_article_cache_path):
    all_fetched_articles = []
    for fname in tqdm([
        '../data/articles/sfchron-cc-fetched-articles.jsonl',
        '../data/articles/sfchron-cc-fetched-articles-2.jsonl.gz',
        '../data/articles/sfchron-fetched-articles.jsonl', 
    ]):
        if fname.endswith('.gz'):
            f = gzip.open(fname)
        else:
            f = open(fname)
        lines = list(jsonlines.Reader(f))
        all_fetched_articles += lines

    all_fetched_articles_df = pd.DataFrame(all_fetched_articles)
    all_fetched_articles_df['path'] = all_fetched_articles_df['article_url'].apply(parse_path)
    all_fetched_articles_df = all_fetched_articles_df.drop_duplicates('article_text')
    all_fetched_articles_df['article_publish_date'] = (
        pd.to_datetime(all_fetched_articles_df['article_publish_date'], errors='coerce', utc=True)
    )
    (all_fetched_articles_df
     [['article_text', 'article_publish_date', 'path']]
     .to_csv(fetched_article_cache_path, compression='gzip')
    )
else:
    all_fetched_articles_df = pd.read_csv(fetched_article_cache_path, index_col=0)
    all_fetched_articles_df['article_publish_date'] = pd.to_datetime(all_fetched_articles_df['article_publish_date'])

# Problem Setup

In [35]:
import os 
import gdown
import re 

In [36]:
full_meeting_row_df = pd.read_csv('cache/2023-04-13__full_meeting_row_df.csv')
full_meeting_row_df = full_meeting_row_df.assign(text=lambda df: df['Name'] + '. ' + df['Title']).drop(columns=['Name', 'Title'])
full_meeting_row_df['date'] = pd.to_datetime(full_meeting_row_df['date'])
all_candidate_articles = pd.read_json('cache/2023-04-19__candidate-articles-filtered-by-keyword.jsonl', lines=True)

In [42]:
output = 'cache/gold-matching.csv'
if not os.path.exists(output):
    url = 'https://docs.google.com/spreadsheets/d/1T319_gMlJeMcl0T56aeTGjm12havON379fP0W2aJmZA/edit?usp=sharing'
    gdown.download(url, output, quiet=True)

first_round_output_file = 'cache/y-true-matched-df.csv.gz'
if not os.path.exists(first_round_output_file):
    first_round_annotation_df = (
        pd.read_excel(output)
        .rename(columns=lambda x: re.sub('\s+', ' ', x.replace('\n', ' ')))
        .rename(columns={'text': 'article', 'City Council Meeting Date': 'meeting_date'})
        .assign(method='full_label_first_round')
    )
    
    sfbos_first_round_annotation_df = (
        first_round_annotation_df.loc[lambda df: df['City Council Meeting Minute URL'].str.contains('sfbos') == True]
    )
    sfbos_first_round_annotation_df['meeting_id'] = sfbos_first_round_annotation_df['Page/paragraph'].str.split(' ').str.get(-1)
    sfbos_first_round_annotation_df = sfbos_first_round_annotation_df.loc[lambda df: df['meeting_id'].str.len() == 6]
    sfbos_first_round_annotation_df['meeting_id'] = sfbos_first_round_annotation_df['meeting_id'].astype(int)
    sfbos_first_round_annotation_df = sfbos_first_round_annotation_df.merge(
        full_meeting_row_df[['File #', 'text']].drop_duplicates(), 
        how='left',
        left_on='meeting_id',
        right_on='File #', 
    ).rename(columns={'text': 'meeting_text'})
    
    sfbos_first_round_annotation_df.to_csv(first_round_output_file, compression='gzip')
else:
    sfbos_first_round_annotation_df = pd.read_csv(first_round_output_file, compression='gzip')

In [38]:
output = 'cache/full-automated-meeting-article-matching__annotated.xlsx'
end_to_end_annotated_df = pd.read_excel(output)

In [39]:
end_to_end_annotated_df = end_to_end_annotated_df.loc[lambda df: df['Label'].notnull()]

In [40]:
all_candidate_articles = pd.read_json('cache/2023-04-19__candidate-articles-filtered-by-keyword.jsonl', lines=True)

# Get Embeddings to Perform Comparison

In [ ]:
### re-run this using SBERT
all_text_df = pd.concat([
    full_meeting_row_df['text'],
    all_fetched_articles_df['article_text']
])
all_text_df.to_csv('cache/all-text-to-score-sbert.csv')

In [ ]:
## score using SBERT (for server)
from sentence_transformers import SentenceTransformer, util
model = SentenceTransformer('microsoft/mpnet-base')
df = pd.read_csv('all-text-to-score-sbert.csv.gz')

In [68]:
# score using OpenAI
import openai

openai.api_key_path = "/Users/spangher/.openai-isi-key.txt"

In [ ]:
response = openai.Embedding.create(
        input=all_text_df.iloc[145013],
        model="text-embedding-ada-002"
    )

In [ ]:
all_openai_embeddings = []

for text in tqdm(all_text_df, total=len(all_text_df)):
    response = openai.Embedding.create(
        input=text,
        model="text-embedding-ada-002"
    )
    embeddings = response['data'][0]['embedding']
    all_openai_embeddings.append(embeddings)

# All articles

In [43]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [46]:
tfidf = TfidfVectorizer(min_df=.001, max_df=.5, stop_words='english')
all_vecs = tfidf.fit_transform(
    pd.concat([
        full_meeting_row_df['text'],
        all_fetched_articles_df['article_text']
    ]).fillna('')
)

In [47]:
article_vecs = all_vecs[len(full_meeting_row_df):]

In [48]:
y_true_meeting_vecs = tfidf.transform(sfbos_first_round_annotation_df['meeting_text'])

In [49]:
sims = cosine_similarity(article_vecs, y_true_meeting_vecs)

In [50]:
sim_df = pd.DataFrame(
    sims, 
    columns=sfbos_first_round_annotation_df['meeting_id'], 
    index=all_fetched_articles_df['path']
)

In [51]:
from unidecode import unidecode

In [52]:
y_true_urls =  (
    sfbos_first_round_annotation_df
         .set_index('meeting_id')['News Article URL']
         .apply(parse_path)
)

In [405]:
(sim_df.idxmax(axis=0) == y_true_urls).mean()

0.16071428571428573

In [123]:
### test using embeddings

In [124]:
all_emb_vecs = np.loadtxt('cache/all-MiniLM-L6-v2-sent-embs.txt')

In [126]:
all_emb_vecs.shape 

(243997, 384)

In [ ]:
article_emb_vecs = all_emb_vecs[]

# Articles filtered by keyword

In [456]:
candidate_rough_df = (
    all_fetched_articles_df
         .loc[lambda df: 
              df['article_text'].str.contains('Board of Supervisors', case=False) |
              df['article_text'].str.contains('supervisor', case=False) 
         ]
)

In [414]:
l1_tfidf = TfidfVectorizer(min_df=.001, max_df=.5, stop_words='english')
all_vecs = l1_tfidf.fit_transform(
    pd.concat([
        full_meeting_row_df['text'].sample(10_000),
#         all_candidate_articles['summary_text']
        candidate_rough_df['article_text']
    ]).fillna('')
)

candidate_vecs = all_vecs[10_000:]
y_true_meeting_vecs = l1_tfidf.transform(sfbos_first_round_annotation_df['meeting_text'])
sims = cosine_similarity(candidate_vecs, y_true_meeting_vecs)
sims_1_df = pd.DataFrame(
    sims, 
#     index=all_candidate_articles['article_url'].apply(parse_path), 
    index=candidate_rough_df['article_url'].apply(parse_path), 
    columns=sfbos_first_round_annotation_df['File #']
)

(sims_1_df.idxmax(axis=0) == y_true_urls).mean()

0.2857142857142857

# Articles l2

In [319]:
from sklearn.linear_model import LogisticRegressionCV
from sklearn.pipeline import Pipeline

In [311]:
output = 'cache/word-matching-annotated-meeting.xlsx'
l2_output_1 = pd.read_excel(output)
output = 'cache/lr-annotated-meeting.xlsx'
l2_output_2 = pd.read_excel(output)
l2_output = pd.concat([l2_output_1, l2_output_2])

In [415]:
pos_examples = (
    l2_output
        .loc[lambda df: df['Meeting'] == 'SFBOS']
        [['article', 'Label']]
)
neg_examples = (
    l2_output
        .loc[lambda df: df['Meeting'] != 'SFBOS']
        .loc[lambda df: df['Label'].notnull()]
        .assign(Label=0)
        [['article', 'Label']]
)
train_df = pd.concat([pos_examples, neg_examples])

In [416]:
l2_pipe = Pipeline([
    ('cv', TfidfVectorizer(min_df=.01, max_df=.5, stop_words='english')),
    ('lr', LogisticRegressionCV())
])

In [417]:
l2_pipe.fit(train_df['article'], y=train_df['Label'])

Pipeline(steps=[('cv',
                 TfidfVectorizer(max_df=0.5, min_df=0.01,
                                 stop_words='english')),
                ('lr', LogisticRegressionCV())])

In [418]:
y_probs = l2_pipe.predict_proba(candidate_rough_df['article_text'])

In [419]:
pd.Series(y_probs.argmax(axis=1)).value_counts()

0    10797
1     2919
dtype: int64

In [ ]:
candidate_rough_df['l2_prob'] = y_probs[:, 1]

In [459]:
l2_filtered_articles = candidate_rough_df.loc[lambda df: df['l2_prob'] > .1]

In [ ]:
l2_tfidf = TfidfVectorizer(min_df=.001, max_df=.5, stop_words='english')
all_vecs = l2_tfidf.fit_transform(
    pd.concat([
        full_meeting_row_df['text'].sample(10_000),
#         l2_filtered_articles['summary_text'],
        l2_filtered_articles['article_text'],        
    ]).fillna('')
)

candidate_vecs = all_vecs[10_000:]
y_true_meeting_vecs = l2_tfidf.transform(sfbos_first_round_annotation_df['meeting_text'])
l2_filtered_articles['path'] = l2_filtered_articles['article_url'].apply(parse_path)
sims = cosine_similarity(candidate_vecs, y_true_meeting_vecs)
sims_2_df = pd.DataFrame(
    sims, 
    index=l2_filtered_articles['path'], 
#     index=candidate_rough_df['article_url'].apply(parse_path), 
    columns=sfbos_first_round_annotation_df['File #']
)

(sims_2_df.idxmax(axis=0) == y_true_urls).mean()

In [471]:
from datetime import timedelta
def get_similar_articles(meeting_text, meeting_date, article_df, k=4, window=30, tfidf=None):
    query_vec = tfidf.transform([meeting_text])
    date_limited_article_df = (
        article_df
            .loc[lambda df: df['article_publish_date'].notnull()]
            .loc[lambda df: df['article_publish_date'].dt.date > meeting_date.date() - timedelta(days=window)]
            .loc[lambda df: df['article_publish_date'].dt.date <= meeting_date.date() + timedelta(days=window)]
#             .loc[lambda df: df['committee'] == 'Board of Supervisors']
    )
    if len(date_limited_article_df) == 0:
        return None 

    key_vecs = date_limited_article_df['article_text'].pipe(tfidf.transform)
    sims = cosine_similarity(query_vec, key_vecs).flatten()
    date_limited_article_df['sims'] = sims
    return date_limited_article_df.sort_values('sims', ascending=False).iloc[:k]

In [515]:
mapping = {}
for idx, (meeting_id, meeting_text, meeting_date) in sfbos_first_round_annotation_df[['meeting_id', 'meeting_text', 'meeting_date']].iterrows():
    t = get_similar_articles(meeting_text, meeting_date, l2_filtered_articles, k=100, window=30, tfidf=l2_tfidf)
    mapping[meeting_id] = t['path'].tolist()

In [516]:
(pd.Series(mapping).str.get(0)
 .to_frame('y_pred')
 .merge(y_true_urls.to_frame('y_true'), right_index=True, left_index=True)
 .pipe(lambda df: df['y_pred'] == df['y_true']).mean()
)

0.5535714285714286

# L3

In [517]:
(pd.Series(mapping)#.str.get(0)
 .to_frame('y_pred')
 .merge(y_true_urls.to_frame('y_true'), right_index=True, left_index=True)
 .apply(lambda x: x['y_true'] in x['y_pred'], axis=1)
#  .pipe(lambda df: df['y_pred'] == df['y_true']).mean()
).mean()

0.7321428571428571

# BMP20

In [1]:
import scipy

In [4]:
from retriv import SearchEngine
collection = [
  {"id": "doc_1", "text": "Generals gathered in their masses"},
  {"id": "doc_2", "text": "Just like witches at black masses"},
  {"id": "doc_3", "text": "Evil minds that plot destruction"},
  {"id": "doc_4", "text": "Sorcerer of death's construction"},
]

In [15]:
article_collection = (all_fetched_articles_df
 [['path', 'article_text']]
 .rename(columns={'path': 'id', 'article_text': 'text'})
 .to_dict(orient='rows')
)

/var/folders/xh/qnyq7yzj0r328_7hnb7pgxth0000gp/T/ipykernel_44677/1241168551.py:1: FutureWarning: Using short name for 'orient' is deprecated. Only the options: ('dict', list, 'series', 'split', 'records', 'index') will be used in a future version. Use one of the above to silence this warning.
  article_collection = (all_fetched_articles_df


In [16]:
se = SearchEngine("article-collection").index(article_collection)

Building inverted index: 100%|██████████████████████████████████████████████████| 274498/274498 [00:23<00:00, 11598.53it/s]


In [24]:
top_docs = se.search("witches masses", return_docs=False, cutoff=len(article_collection))

In [ ]:
all_fetched_articles_df.merge(pd.Series(top_docs).to_frame('score'), right_index=True, left_on='path')

In [57]:
from datetime import timedelta
def get_similar_articles_bmp_20(meeting_text, meeting_date, article_df, k=4, window=30, se=None):
    top_docs = se.search(meeting_text, return_docs=False, cutoff=200_000)
    top_article_df = (
        article_df
            .merge(pd.Series(top_docs).to_frame('score'), right_index=True, left_on='path')
    )
    date_limited_article_df = (
        top_article_df
            .loc[lambda df: df['article_publish_date'].notnull()]
            .loc[lambda df: df['article_publish_date'].dt.date > meeting_date.date() - timedelta(days=window)]
            .loc[lambda df: df['article_publish_date'].dt.date <= meeting_date.date() + timedelta(days=window)]
#             .loc[lambda df: df['committee'] == 'Board of Supervisors']
    )
    if len(date_limited_article_df) == 0:
        return None 
    
    return date_limited_article_df.sort_values('score', ascending=False).iloc[:k]

In [59]:
from tqdm.auto import tqdm

In [ ]:
mapping = {}
for idx, (meeting_id, meeting_text, meeting_date) in tqdm(
    sfbos_first_round_annotation_df[['meeting_id', 'meeting_text', 'meeting_date']].iterrows(), total=len(sfbos_first_round_annotation_df)
):
    t = get_similar_articles_bmp_20(
        meeting_text, meeting_date, all_fetched_articles_df, k=4, window=30, se=se)
    mapping[meeting_id] = t['path'].tolist()

In [65]:
(pd.Series(mapping)#.str.get(0)
 .to_frame('y_pred')
 .merge(y_true_urls.to_frame('y_true'), right_index=True, left_index=True)
 .apply(lambda x: x['y_true'] in x['y_pred'], axis=1)
#  .pipe(lambda df: df['y_pred'] == df['y_true']).mean()
).mean()

0.6071428571428571

# Dense Retriever

In [ ]:
from retriv import DenseRetriever

dr = DenseRetriever(
  index_name="dense-index",
  model="sentence-transformers/all-MiniLM-L6-v2",
  normalize=True,
  max_length=128,
  use_ann=True,
)

In [ ]:
SearchEngine(model=)